In [ ]:
# imports
import time
import pandas as pd
import requests
from datetime import datetime
from riotwatcher import LolWatcher, ApiError
from pathlib import Path

# repository root: prefer file-based parent when available, else step up from CWD
try:
    ROOT = Path(__file__).resolve().parents[1]
except NameError:
    _cwd = Path().resolve()
    ROOT = _cwd.parent if _cwd.name == "src" else _cwd

In [ ]:
watcher = None
_active_key = None  # tracks the key currently loaded into watcher

# parse the data we need from match jsons
def parse_match_data(match_json):
    participants = match_json["info"]["participants"]
    red, blue = {}, {}
    for p in participants:
        champ = p["championName"]
        pos = p.get("teamPosition", "").lower()
        team = p["teamId"]
        if team == 100:
            blue[pos] = champ
        elif team == 200:
            red[pos] = champ
    blue_team = next((t for t in match_json["info"]["teams"] if t["teamId"] == 100), None)
    blue_win = 1 if blue_team and blue_team["win"] else 0
    return {
        "red_top": red.get("top"),
        "red_jg": red.get("jungle"),
        "red_mid": red.get("middle"),
        "red_adc": red.get("bottom"),
        "red_sup": red.get("utility"),
        "blue_top": blue.get("top"),
        "blue_jg": blue.get("jungle"),
        "blue_mid": blue.get("middle"),
        "blue_adc": blue.get("bottom"),
        "blue_sup": blue.get("utility"),
        "winner": blue_win,
    }

# reads api key from riot_api_key.txt
def load_key():
    try:
        local_path = Path(__file__).resolve().parent / "riot_api_key.txt"
    except NameError:
        local_path = Path.cwd() / "riot_api_key.txt"

    if local_path.exists():
        return local_path.read_text().strip()

    src_path = Path.cwd() / "src" / "riot_api_key.txt"
    if src_path.exists():
        return src_path.read_text().strip()

    key_path = Path(ROOT) / "riot_api_key.txt"
    if key_path.exists():
        return key_path.read_text().strip()

    raise ValueError("Riot API key not found. Please create 'riot_api_key.txt' in the src folder with your API key.")


def make_watcher():
    global _active_key
    key = load_key()
    _active_key = key
    print(f"Loaded API key: {key[:10]}... (length: {len(key)})")
    return LolWatcher(key)


def wait_for_new_key():
    """Block until riot_api_key.txt is updated with a new key, then reload the watcher."""
    global watcher, _active_key
    print("\n=== API key expired or invalid ===")
    print("Paste your new key into riot_api_key.txt and save it. Checking every 15 seconds...")
    while True:
        time.sleep(15)
        try:
            new_key = load_key()
        except Exception as e:
            print(f"  Still waiting (could not read key file: {e})")
            continue
        if new_key != _active_key:
            print(f"New key detected ({new_key[:10]}...). Resuming collection.")
            _active_key = new_key
            watcher = LolWatcher(new_key)
            return
        print("  Key unchanged, still waiting...")


# fetch a single match from the API, blocking on rate limits and pausing on key expiry
def fetch_match(region, mid, total_timeout=120):
    global watcher
    if watcher is None:
        watcher = make_watcher()
    start = time.time()
    while time.time() - start < total_timeout:
        try:
            return watcher.match.by_id(region, mid)
        except ApiError as e:
            if e.response.status_code == 401:
                wait_for_new_key()
                start = time.time()  # reset timeout after key renewal
                continue
            elif e.response.status_code == 429:
                retry_after = int(e.response.headers.get("Retry-After", 2))
                print(f"Rate limit hit. Waiting {retry_after}s...")
                time.sleep(retry_after + 1)
                continue
            else:
                print(f"API error {e.response.status_code} for {mid}")
                time.sleep(1)
        except Exception as e:
            print(f"Error fetching {mid}: {e}")
            time.sleep(1)
    print(f"Skipped match {mid} after {int(time.time() - start)}s.")
    return None


# builds dataframe and writes output to files
def build_dataframe(region, puuid_list, output_file="matches_latest.csv", max_players=None):
    global watcher
    watcher = make_watcher()
    all_rows, seen_matches = [], set()
    total_matches = 0

    out_path = Path(output_file)
    if not out_path.is_absolute():
        out_path = Path(ROOT) / "data" / "ryan_match_data" / out_path
    out_path.parent.mkdir(parents=True, exist_ok=True)

    print(f"Writing to {out_path}")
    print(f"Starting build_dataframe with {len(puuid_list)} players (limited to {max_players} if set)")

    if max_players:
        puuid_list = puuid_list[:max_players]
        print(f"Limited to first {max_players} players")

    try:
        for i, entry in enumerate(puuid_list, start=1):
            puuid = entry.get("puuid") if isinstance(entry, dict) else None
            if not puuid:
                try:
                    summ = watcher.summoner.by_id(PLATFORM, entry["summonerId"])
                    puuid = summ["puuid"]
                except Exception as e:
                    print(f"Error getting puuid for player {i}: {e}")
                    continue

            print(f"\nPlayer {i}/{len(puuid_list)}: {puuid[:10]}")
            match_ids, start_idx = [], 0

            while True:
                try:
                    ids = watcher.match.matchlist_by_puuid(region, puuid, start=start_idx, count=100)
                    if not ids:
                        break
                    match_ids.extend(ids)
                    start_idx += 100
                    time.sleep(0.4)
                except ApiError as e:
                    if e.response.status_code == 429:
                        retry_after = int(e.response.headers.get("Retry-After", 2))
                        print(f"Rate limit. Sleeping {retry_after}s...")
                        time.sleep(retry_after + 1)
                        continue
                    if e.response.status_code == 401:
                        wait_for_new_key()
                        continue
                    print(f"Error fetching match list for {puuid[:10]}: {e}")
                    break
                except Exception as e:
                    print(f"Unexpected error fetching match list for {puuid[:10]}: {e}")
                    break

            print(f"Player {i}: Retrieved {len(match_ids)} matches.")
            for j, mid in enumerate(match_ids, start=1):
                if mid in seen_matches:
                    continue
                data = fetch_match(region, mid, total_timeout=100)
                if not data:
                    continue
                try:
                    row = parse_match_data(data)
                    row["match_id"] = mid
                    all_rows.append(row)
                    seen_matches.add(mid)
                    total_matches += 1
                    if j % 10 == 0 or total_matches % 50 == 0:
                        print(f"Player {i}, Match {j}/{len(match_ids)} ({mid}) added ({total_matches} total).")
                    if total_matches % 100 == 0:
                        pd.DataFrame(all_rows).to_csv(str(out_path), index=False)
                        print(f"Checkpoint: saved {total_matches} matches.")
                except Exception as e:
                    print(f"Parse error for {mid}: {e}")
                time.sleep(0.4)
            time.sleep(1)

    finally:
        # always flush to disk on exit, whether normal, interrupted, or exception
        if all_rows:
            pd.DataFrame(all_rows).to_csv(str(out_path), index=False)
            print(f"Saved {len(all_rows)} matches to {out_path}")
        else:
            print("No matches collected.")

    return pd.DataFrame(all_rows)


# finds all top 300 players
def get_challenger_entries(region="na1", queue="RANKED_SOLO_5x5"):
    api_key = load_key()
    headers = {"X-Riot-Token": api_key}
    url = f"https://{region}.api.riotgames.com/lol/league/v4/challengerleagues/by-queue/{queue}"
    print(f"Fetching challenger entries from {url}")
    r = requests.get(url, headers=headers)
    print(f"Response status: {r.status_code}")
    if r.status_code != 200:
        print(f"Response text: {r.text}")
    r.raise_for_status()
    return r.json().get("entries", [])


REGION = "americas"
PLATFORM = "na1"

watcher = make_watcher()
entries = get_challenger_entries(region=PLATFORM)
print(f"Retrieved {len(entries)} Challenger entries from {PLATFORM}")
df = build_dataframe(REGION, entries, output_file=f"matches_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}.csv")
print(df.head())

In [ ]:
# Post-match data extraction: parse Riot match JSONs and write participant-level stats to CSV
# Put your match JSON files in ../data/raw_matches/ (relative to this notebook). The script will create the directory if missing.

import os
import json
from pathlib import Path
from typing import Dict, Any, List

try:
    import pandas as pd
except Exception:
    pd = None

RAW_DIR = Path("../data/raw_matches").resolve()
OUT_FILE = Path("../data/post_match_stats.csv").resolve()

RAW_DIR.mkdir(parents=True, exist_ok=True)


def safe_get(d: Dict[str, Any], *keys, default=None):
    """Safely walk nested dict keys; returns default if any key missing."""
    cur = d
    for k in keys:
        if not isinstance(cur, dict):
            return default
        cur = cur.get(k, default)
        if cur is default:
            return default
    return cur


def parse_match_json(match_json: Dict[str, Any]) -> List[Dict[str, Any]]:
    """Parse a Riot-style match JSON and return a list of participant-level flattened stats.

    The function is defensive: it looks for common keys from Match V5 and older schemas and
    uses .get() fallbacks so it won't crash if a key is missing.
    """
    # Try to find match id
    match_id = safe_get(match_json, "metadata", "matchId") or match_json.get("gameId") or match_json.get("matchId")

    info = match_json.get("info") or match_json.get("game") or match_json

    participants = info.get("participants") or info.get("participant", [])
    teams = info.get("teams") or []

    # Build a map from teamId -> aggregated team objectives and wins
    team_map = {}
    for t in teams:
        # Team id might be 'teamId' (100/200) or 'teamId' absent
        team_id = t.get("teamId") or t.get("teamIdNumber") or None
        # Try to pull objectives safely
        objectives = t.get("objectives") if isinstance(t.get("objectives"), dict) else t.get("objectives", {})
        baron_kills = safe_get(objectives, "baron", "kills") if objectives else t.get("baronKills")
        dragon_kills = safe_get(objectives, "dragon", "kills") if objectives else t.get("dragonKills")
        tower_kills = safe_get(objectives, "tower", "kills") if objectives else t.get("towerKills") or t.get("towersDestroyed")
        inhibitor_kills = safe_get(objectives, "inhibitor", "kills") if objectives else t.get("inhibitorKills")

        team_map[team_id] = {
            "team_win": t.get("win") if "win" in t else (t.get("winner") if "winner" in t else None),
            "team_baron_kills": baron_kills if baron_kills is not None else 0,
            "team_dragon_kills": dragon_kills if dragon_kills is not None else 0,
            "team_tower_kills": tower_kills if tower_kills is not None else 0,
            "team_inhibitor_kills": inhibitor_kills if inhibitor_kills is not None else 0,
            "team_id": team_id,
        }

    rows = []
    for p in participants:
        # Participant stats might be nested or at top-level
        stats = p.get("stats") if isinstance(p.get("stats"), dict) else p

        participant_id = p.get("participantId") or p.get("puuid") or p.get("summonerId") or p.get("id")
        team_id = p.get("teamId") or p.get("team") or None

        kills = stats.get("kills") or stats.get("killsCount") or 0
        deaths = stats.get("deaths") if stats.get("deaths") is not None else stats.get("death", 0)
        assists = stats.get("assists") or 0
        kda = (kills + assists) / max(1, deaths) if deaths is not None else (kills + assists)

        total_damage_to_champ = stats.get("totalDamageDealtToChampions") or stats.get("damageDealtToChampions") or stats.get("totalDamageDealt") or 0
        total_damage_taken = stats.get("totalDamageTaken") or stats.get("damageTaken") or 0
        damage_to_buildings = stats.get("damageDealtToBuildings") or stats.get("damageDealtToTurrets") or stats.get("damageDealtToObjectives") or 0

        gold_earned = stats.get("goldEarned") or stats.get("gold") or 0
        total_minions_killed = stats.get("totalMinionsKilled") or stats.get("minionsKilled") or 0
        neutral_minions = stats.get("neutralMinionsKilled") or stats.get("neutralMinionsKilledTeamJungle") or 0
        cs = total_minions_killed + neutral_minions

        vision_score = stats.get("visionScore") or stats.get("vision") or 0
        time_ccing_others = stats.get("timeCCingOthers") or stats.get("timeCCingOthersTotal") or 0

        champ_level = stats.get("champLevel") or stats.get("level") or None

        # common kill types
        double_kills = stats.get("doubleKills") or 0
        triple_kills = stats.get("tripleKills") or 0
        quadra_kills = stats.get("quadraKills") or 0
        penta_kills = stats.get("pentaKills") or 0

        # turret / building contributions
        turret_kills = stats.get("turretKills") or stats.get("turretsDestroyed") or 0
        damage_self_mitigated = stats.get("damageSelfMitigated") or 0

        participant_row = {
            "match_id": match_id,
            "participant_id": participant_id,
            "summonerName": p.get("summonerName") or p.get("displayName") or None,
            "puuid": p.get("puuid") or p.get("player") or None,
            "championId": p.get("championId") or p.get("champion" ) or stats.get("championId") or stats.get("championName"),
            "championName": p.get("championName") or stats.get("championName"),
            "team_id": team_id,
            "win": stats.get("win") if "win" in stats else None,

            "kills": kills,
            "deaths": deaths,
            "assists": assists,
            "kda": kda,

            "totalDamageToChampions": total_damage_to_champ,
            "totalDamageTaken": total_damage_taken,
            "damageToBuildings": damage_to_buildings,
            "damageSelfMitigated": damage_self_mitigated,

            "goldEarned": gold_earned,
            "cs": cs,
            "totalMinionsKilled": total_minions_killed,
            "neutralMinionsKilled": neutral_minions,

            "visionScore": vision_score,
            "timeCCingOthers": time_ccing_others,
            "champLevel": champ_level,

            "doubleKills": double_kills,
            "tripleKills": triple_kills,
            "quadraKills": quadra_kills,
            "pentaKills": penta_kills,

            "turretKills": turret_kills,
        }

        # Enrich with team-level objective counts when available
        team_info = team_map.get(team_id) or team_map.get(100 if team_id is None else None) or {}
        participant_row.update({
            "team_win": team_info.get("team_win"),
            "team_baron_kills": team_info.get("team_baron_kills", 0),
            "team_dragon_kills": team_info.get("team_dragon_kills", 0),
            "team_tower_kills": team_info.get("team_tower_kills", 0),
            "team_inhibitor_kills": team_info.get("team_inhibitor_kills", 0),
        })

        rows.append(participant_row)

    return rows


def collect_matches_to_csv(raw_dir: Path = RAW_DIR, out_file: Path = OUT_FILE):
    """Scan a directory of JSON files (match exports), parse them, and append to a CSV.

    - If the CSV exists, new rows will be concatenated and duplicates (match_id + participant_id) removed.
    - Works with Riot Match-v5 exports that have 'metadata' and 'info' top-level keys and older layouts.
    """
    all_rows = []
    for fp in sorted(raw_dir.glob("*.json")):
        try:
            with open(fp, "r", encoding="utf-8") as f:
                j = json.load(f)
        except Exception as e:
            print(f"Skipping {fp.name}: failed to load JSON ({e})")
            continue

        try:
            rows = parse_match_json(j)
            all_rows.extend(rows)
        except Exception as e:
            print(f"Error parsing {fp.name}: {e}")

    if not all_rows:
        print("No match rows found in raw dir. Drop JSON match files into:", raw_dir)
        return None

    # If pandas available, use it for convenient CSV handling
    if pd is not None:
        df_new = pd.DataFrame(all_rows)
        if out_file.exists():
            df_old = pd.read_csv(out_file)
            df = pd.concat([df_old, df_new], ignore_index=True)
            # Drop exact duplicates by match+participant
            df.drop_duplicates(subset=["match_id", "participant_id"], inplace=True)
        else:
            df = df_new
        out_file.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(out_file, index=False)
        print(f"Wrote {len(df)} participant rows to {out_file}")
        return df
    else:
        # Fallback: write CSV manually
        import csv

        out_file.parent.mkdir(parents=True, exist_ok=True)
        headers = list(all_rows[0].keys())
        existing = set()
        if out_file.exists():
            try:
                with open(out_file, "r", newline='', encoding="utf-8") as f:
                    reader = csv.DictReader(f)
                    for r in reader:
                        existing.add((r.get("match_id"), r.get("participant_id")))
            except Exception:
                existing = set()

        appended = 0
        with open(out_file, "a", newline='', encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=headers)
            if f.tell() == 0:
                writer.writeheader()
            for r in all_rows:
                key = (r.get("match_id"), r.get("participant_id"))
                if key in existing:
                    continue
                writer.writerow(r)
                appended += 1
        print(f"Appended {appended} rows to {out_file}")
        return out_file


# Example usage
if __name__ == "__main__":
    print("Scanning for match JSONs in:", RAW_DIR)
    collect_matches_to_csv()

# Small helper: run in notebook to collect immediately
print('Utility functions defined. To use in this notebook run: collect_matches_to_csv()')


Scanning for match JSONs in: C:\Users\Laser\cs163-group18-winfactors\data\raw_matches
No match rows found in raw dir. Drop JSON match files into: C:\Users\Laser\cs163-group18-winfactors\data\raw_matches
Utility functions defined. To use in this notebook run: collect_matches_to_csv()


In [3]:
# Test the API key with a simple request
# Run this cell to verify your Riot API key works before running the main pipeline.

import requests

try:
    api_key = load_key()
    print(f"Testing key: {api_key[:10]}... (length: {len(api_key)})")
    headers = {"X-Riot-Token": api_key}
    # Test with a known summoner (change 'Doublelift' to any valid name if needed)
    test_url = "https://na1.api.riotgames.com/lol/summoner/v4/summoners/by-name/Doublelift"
    r = requests.get(test_url, headers=headers)
    print(f"Response status: {r.status_code}")
    if r.status_code == 200:
        data = r.json()
        print(f"Success! Summoner: {data.get('name')}, Level: {data.get('summonerLevel')}")
    else:
        print(f"Failed: {r.text}")
except Exception as e:
    print(f"Error testing key: {e}")

Testing key: RGAPI-186e... (length: 42)
Response status: 401
Failed: {"status":{"message":"Unknown apikey","status_code":401}}


In [ ]:
# Run the post-match data collection with sample data
# This will parse the sample match JSON and create data/post_match_stats.csv

collect_matches_to_csv()